# CSCI6032 Homework 8

**Instructions**

* **Insert all code, plots, results, and discussion** into this Jupyter Notebook.
* Your homework should be submitted as a **single Jupyter Notebook** (.ipynb file) to **Blackboard**.
* While working, you can use Visual Studio Code, Jupyter Notebook or Google Colab.  If using Google Colab, export it as a Jupyter Notebook (.ipynb).

You can answer mathematical questions either by:
* using LaTeX in a markdown cell, or
* handwriting your answer, then photographing or scanning it and using an AI/LLM to convert it into markdown and LaTeX, checking the conversion for accuracy before submitting.


**Problems 1 to 3 use only numpy** (plus pandas to load data and matplotlib to plot).  Do not use PyTorch, autograd, or
scikit-learn there: the point is to compute every gradient yourself.  **Problem 4 uses PyTorch**, to check your numpy code against it.

Run all of Problem 4 in **one environment**, so that the timings are comparable: either one Google Colab session (PyTorch is preinstalled;
a free GPU is available but not needed) or your own computer (`pip install torch`).  In Colab, upload `p4_data.csv` and
`p3_multiclass.csv` with the Files panel on the left.

---
## Problem 1: Backpropagation by Hand

Lecture 23 derives backpropagation for a network with $L$ layers.  Here you derive it for the small network from lecture 21 and
HW7 Problem 5, then check it on numbers.  The network has two inputs, one hidden layer of $h$ units, and one output unit, with the
logistic function $\sigma$ as every activation:
$$
\bar x \in \mathbb{R}^2 \;\longrightarrow\; \bar z^{[1]} = W^{[1]}\bar x + \bar b^{[1]} \;\longrightarrow\; \bar a^{[1]} = \sigma(\bar z^{[1]})
\;\longrightarrow\; z^{[2]} = W^{[2]}\bar a^{[1]} + b^{[2]} \;\longrightarrow\; \hat y = \sigma(z^{[2]}),
$$
trained on the cross-entropy loss $L = -\big[y\log\hat y + (1-y)\log(1-\hat y)\big]$ for one sample $(\bar x, y)$.  Here $W^{[1]}$ is
$h \times 2$ and $W^{[2]}$ is $1 \times h$.  Vectors are columns.

**(a) Jacobians of the building blocks.**  (Lecture 23, pages 2 to 4.)

1. For $\bar z = W\bar a + \bar b$ with $W$ of size $k \times n$, show that the Jacobian $\frac{\partial \bar z}{\partial \bar a}$ is $W$ and
   that $\frac{\partial \bar z}{\partial \bar b}$ is the $k \times k$ identity matrix.
2. For the elementwise activation $\bar a = \sigma(\bar z)$, show that the Jacobian $\frac{\partial \bar a}{\partial \bar z}$ is the diagonal
   matrix $\operatorname{diag}\big(\sigma'(\bar z)\big)$.  Explain why multiplying a vector by this matrix is the same as taking its
   elementwise (Hadamard) product $\odot$ with $\sigma'(\bar z)$.
3. For a scalar loss $L$, the chain rule gives $\frac{\partial L}{\partial \bar a} = \Big(\frac{\partial \bar z}{\partial \bar a}\Big)^T
   \frac{\partial L}{\partial \bar z} = W^T\frac{\partial L}{\partial \bar z}$.  Where does the transpose come from?

(4 points)

**(b) The gradient with respect to a matrix.**  (Lecture 23, page 4.)

1. For column vectors $\bar u$ and $\bar a$, the quantity $\bar u^T W \bar a$ is a scalar.  Using the cyclic property of the trace
   (Homework 1) and $\nabla_X \operatorname{tr}(AX) = A^T$, show that $\nabla_W\big(\bar u^T W \bar a\big) = \bar u\,\bar a^T$.
2. Let $\bar\delta = \frac{\partial L}{\partial \bar z}$ for $\bar z = W\bar a + \bar b$.  Use (1) to explain why
   $\frac{\partial L}{\partial W} = \bar\delta\,\bar a^T$, an outer product with the same shape as $W$.

(4 points)

**(c) Backpropagation for one hidden layer.**  Write $\delta^{[2]} = \frac{\partial L}{\partial z^{[2]}}$ and
$\bar\delta^{[1]} = \frac{\partial L}{\partial \bar z^{[1]}}$.

1. Show that $\delta^{[2]} = \hat y - y$.  (You showed this for one neuron in HW7 Problem 4.)
2. Using (a), show that $\bar\delta^{[1]} = \big(W^{[2]T}\delta^{[2]}\big) \odot \sigma'(\bar z^{[1]})$.
3. Using (b), find $\frac{\partial L}{\partial W^{[2]}}$, $\frac{\partial L}{\partial b^{[2]}}$, $\frac{\partial L}{\partial W^{[1]}}$, and
   $\frac{\partial L}{\partial \bar b^{[1]}}$, and write the four gradient descent update rules.
4. Give the shape of every quantity above ($\bar x$, $\bar z^{[1]}$, $\bar a^{[1]}$, $z^{[2]}$, $\hat y$, $\delta^{[2]}$, $\bar\delta^{[1]}$, and
   each gradient) for $h$ hidden units, and check that every product in your formulas has matching dimensions.
5. Compare your results with equations (15), (24), and (25) of lecture 23, which average over $m$ samples.

(5 points)

**(d) One step by hand.**  Take $h = 2$ and
$$
W^{[1]} = \begin{bmatrix} 1 & -1 \\ -1 & 1 \end{bmatrix}, \quad \bar b^{[1]} = \begin{bmatrix} 0 \\ 0 \end{bmatrix}, \quad
W^{[2]} = \begin{bmatrix} 1 & 1 \end{bmatrix}, \quad b^{[2]} = -1, \quad \bar x = \begin{bmatrix} 1 \\ 0 \end{bmatrix}, \quad y = 1, \quad \alpha = 1.
$$

1. Compute the forward pass: $\bar z^{[1]}$, $\bar a^{[1]}$, $z^{[2]}$, $\hat y$, and $L$.  (Keep four decimal places.)
2. Compute $\delta^{[2]}$, $\bar\delta^{[1]}$, and all four gradients.
3. Take one gradient descent step.  Give the new parameters, and the new $\hat y$ and $L$.  Did the step help?
4. Confirm your numbers with a few lines of numpy.

(5 points)

**(e) Backpropagation for $L$ layers.**  (Lecture 23, pages 5 to 12.)  Now let the network have $L$ layers, with $n^{[\ell]}$ units in layer $\ell$,
$\bar a^{[0]} = \bar x$, $\bar z^{[\ell]} = W^{[\ell]}\bar a^{[\ell-1]} + \bar b^{[\ell]}$, and $\bar a^{[\ell]} = \sigma(\bar z^{[\ell]})$, with a
single output unit $\hat y = a^{[L]}$ and the cross-entropy loss of (c).  Write $\bar\delta^{[\ell]} = \frac{\partial L}{\partial \bar z^{[\ell]}}$.

1. Show that $\delta^{[L]} = \hat y - y$ and that for $\ell < L$
   $$\bar\delta^{[\ell]} = \big(W^{[\ell+1]T}\bar\delta^{[\ell+1]}\big) \odot \sigma'(\bar z^{[\ell]}).$$
   Why must $\bar\delta^{[\ell]}$ be computed after $\bar\delta^{[\ell+1]}$, that is, why does the recursion run *backward*?
2. Show that $\frac{\partial L}{\partial W^{[\ell]}} = \bar\delta^{[\ell]}\bar a^{[\ell-1]T}$ and $\frac{\partial L}{\partial \bar b^{[\ell]}} = \bar\delta^{[\ell]}$.
3. Take a network with two hidden layers of three units each, $n^{[0]}, \dots, n^{[3]} = 2, 3, 3, 1$.  Give the shape of every $W^{[\ell]}$, $\bar b^{[\ell]}$,
   $\bar z^{[\ell]}$, $\bar a^{[\ell]}$, and $\bar\delta^{[\ell]}$, and the total number of parameters.  Then write the backward pass as a loop in pseudocode
   (a few lines) that returns all the gradients.
4. Count the multiplications in the forward pass and in the backward pass for one sample, in terms of the layer sizes.  (For layer $\ell$, the forward pass
   multiplies $W^{[\ell]}$ by a vector; the backward pass multiplies $W^{[\ell]T}$ by a vector and forms one outer product.  You can ignore the
   elementwise operations, and the product $W^{[1]T}\bar\delta^{[1]}$, which is never needed.)  How does the cost of the backward pass compare with the forward pass?

(7 points)

**(f) A softmax output layer.**  (Lecture 23, pages 13 to 16.)  For $K$ classes the output layer has $K$ units that compute
$\bar z \in \mathbb{R}^K$ and then
$$a_k = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}, \qquad k = 1, \dots, K,$$
and the label is a one-hot vector $\bar y$ ($y_c = 1$ for the true class $c$, and 0 elsewhere).  The loss is $L = -\sum_k y_k\log a_k = -\log a_c$.

1. Show that $\sum_k a_k = 1$ and that $\frac{\partial a_i}{\partial z_j} = a_i(\delta_{ij} - a_j)$, where $\delta_{ij}$ is 1 if $i = j$ and 0 otherwise, so that the Jacobian is
   $\operatorname{diag}(\bar a) - \bar a\bar a^T$.  Why is this Jacobian not diagonal, unlike the Jacobian of the elementwise sigmoid in (a)(2)?
2. Use the chain rule to show that $\frac{\partial L}{\partial \bar z} = \bar a - \bar y$.  Compare with $\delta^{[2]} = \hat y - y$ in (c)(1).  What does this say about the hidden layers: does
   backpropagation through them change when the output is softmax instead of sigmoid?
3. Show that for $K = 2$ the softmax output $a_1$ equals $\sigma(z_1 - z_2)$, so a two-class softmax is a logistic regression in disguise.
4. For $\bar z = (2, 1, 0)^T$ and true class $c = 2$ (so $\bar y = (0, 1, 0)^T$), compute $\bar a$, $L$, and $\frac{\partial L}{\partial \bar z}$.  Confirm your numbers with a few lines of numpy.
5. Computing $e^{z_k}$ directly overflows for $z_k$ around 1000.  Show that subtracting the same constant $m$ from every $z_k$ leaves $\bar a$ unchanged, and explain why
   choosing $m = \max_k z_k$ avoids overflow.

(7 points)

---
## Problem 2: A Multilayer Perceptron from Scratch for XOR

The file `p4_data.csv` is the noisy XOR data set from HW7 Problem 5, on which logistic regression fails.  Here you build the network
from Problem 1 and train it.

**(a)** Implement a multilayer perceptron (MLP) from scratch using only numpy and standard python libraries (no autograd, PyTorch, or scikit-learn).
It should have two inputs, one hidden layer of $h$ units, and one output unit, with the logistic function $\sigma(z)=1/(1+e^{-z})$ as the
activation of every hidden and output unit.  Train it on the cross-entropy loss by gradient descent, with the gradients computed by backpropagation.  Give it `fit` and `predict` methods.
Include a unit test that confirms that, after training on the four noise-free points

| $x_1$ | $x_2$ | $y$ |
|:-:|:-:|:-:|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

your MLP classifies all four correctly.  (Training a neural network starts from random weights, so fix the random seed in the test,
and try another seed if training gets stuck.) (8 points)

**(b) Check your gradients.**  Before training, check your backpropagation against a numerical derivative.  For every parameter
$\theta$ (every entry of every weight matrix and bias vector), compute
$$\frac{L(\theta + \epsilon) - L(\theta - \epsilon)}{2\epsilon}, \qquad \epsilon = 10^{-5},$$
changing only that one parameter, and compare it with the value from backpropagation for a few samples.  Report the largest relative
difference; with correct gradients it is around $10^{-7}$ or smaller.  Also set your network to the parameters from Problem 1(d) and
confirm that it reproduces your gradients there.  Why is the central difference above more accurate than
$\big(L(\theta + \epsilon) - L(\theta)\big)/\epsilon$? (5 points)

**(c)** Divide the data into a training set and a test set (you may reuse your split from HW7 Problem 5(a)).  Train your MLP on the training set, with $h=4$ hidden units.  Plot the training loss against the epoch. (4 points)

**(d)** Measure the accuracy of your MLP on the test set and compare it with the accuracy of logistic regression from HW7 Problem 5(b), in a table.  Plot the decision
regions of the MLP. (4 points)

**(e)** Train your MLP with $h = 1, 2, 4,$ and $8$ hidden units, using several random initializations for each, and report the test accuracy for each
run.  What happens with $h=1$, and why?  Why do more hidden units help? (5 points)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data from the CSV file
data = pd.read_csv('p4_data.csv')

# Plot the data, one color per class
plt.figure(figsize=(6, 5))
for label, color in ((0, 'tab:blue'), (1, 'tab:orange')):
    points = data[data['Label'] == label]
    plt.scatter(points['Feature_1'], points['Feature_2'], s=10, alpha=0.6, color=color, label=f'class {label}')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.title('p4_data.csv: %d samples' % len(data))
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

data

---
## Problem 3: A Multiclass MLP from Scratch

The file `p3_multiclass.csv` has two features, `Feature_1` and `Feature_2`, and a label in $\{0, 1, 2\}$ (three classes, 600 samples), in the same format as `p4_data.csv`.

**(a)** The cell below loads the data and plots it, with one color per class.  Describe where the classes lie.  Could a model of the form $\bar a = \operatorname{softmax}(W\bar x + \bar b)$,
which has no hidden layer, separate them well?  Explain. (3 points)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

data3 = pd.read_csv('p3_multiclass.csv')

plt.figure(figsize=(6, 5))
for label, color in ((0, 'tab:blue'), (1, 'tab:orange'), (2, 'tab:green')):
    points = data3[data3['Label'] == label]
    plt.scatter(points['Feature_1'], points['Feature_2'], s=10, alpha=0.6, color=color, label=f'class {label}')
plt.xlabel('Feature_1')
plt.ylabel('Feature_2')
plt.title('p3_multiclass.csv: %d samples' % len(data3))
plt.legend()
plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.show()

data3['Label'].value_counts().sort_index()

**(b)** Extend your MLP from Problem 2 to $K$ output units with a softmax output and the cross-entropy loss $-\log a_c$, averaged over the samples.  Keep the logistic function
on the hidden layer.  `fit` takes integer labels $0, \dots, K-1$ and `predict` returns the class with the largest output.  A hidden size of 0 means no hidden layer, that is,
softmax regression.  Compute the softmax in the stable form of 1(f)(5).  Include two tests: the outputs for each sample sum to 1 (try a sample with very large features), and
the central-difference check of 2(b) on your backpropagation, reporting the largest relative difference. (7 points)

**(c)** Split the data into 450 training samples and 150 test samples (fix the random seed).  Train with $h = 8$ hidden units.  Plot the training loss against the epoch, report the test accuracy,
and show the $3\times3$ confusion matrix of the test set. (4 points)

**(d)** Train with $h = 0, 2, 4, 8,$ and $16$ hidden units, with several random initializations for each, and report the test accuracy of each run in a table.  Plot the decision regions for $h = 0$ and for $h = 8$.
What is the accuracy for $h = 0$, and what is its training loss compared with $\ln 3$?  Explain what the model has learned.  Why does a hidden layer fix this? (5 points)

---
## Problem 4: Checking Your Implementation Against PyTorch

PyTorch computes gradients automatically (autograd) and provides layers and optimizers.  Here you use it as a reference to check your numpy implementation from Problems 2 and 3, and to compare their speed.
Here, and only here, you may use PyTorch.  **Run all of this problem in one environment** (one Colab session, or one computer), and do not change it between parts.

**(a) Environment.**  Run the cell below, and state in a markdown cell where you ran this problem (for example "Colab, CPU runtime", "Colab, T4 GPU runtime", or your computer's model).  It sets PyTorch to 64-bit floats,
as numpy uses, so that the differences below come from rounding alone. (2 points)

In [ ]:
import platform
import time

import numpy as np
import torch

torch.set_default_dtype(torch.float64)

print('python', platform.python_version(), '| numpy', np.__version__, '| torch', torch.__version__)
print('processor:', platform.processor() or platform.machine(), '| CPU threads used by torch:', torch.get_num_threads())
print('GPU available:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

**(b) Compare the gradients.**  Build the PyTorch network with the architecture of Problem 3 ($2 \to 8 \to 3$, logistic hidden layer, `torch.nn.Linear` layers, loss `torch.nn.functional.cross_entropy`, which applies the softmax itself),
and copy your numpy weights into it.  (`torch.nn.Linear` stores its weight matrix as (outputs $\times$ inputs), the same as $W^{[\ell]}$ here.)  On the 450 training samples, compute the loss and call `loss.backward()`.  Report the largest absolute difference between the
two losses and between each of your gradients and PyTorch's `.grad`.  Do the same for the sigmoid network of Problem 2 ($2 \to 4 \to 1$, loss `torch.nn.functional.binary_cross_entropy_with_logits`) on the XOR data.
Autograd should agree with your gradients to about $10^{-15}$, while the finite differences of 2(b) agree to only about $10^{-7}$.  Why is the agreement so much better?  What does the finite-difference check have that this one lacks? (5 points)

**(c) Compare the training.**  Train the Problem 3 network in PyTorch from the *same* initial weights as your numpy network, by full-batch gradient descent with `torch.optim.SGD`, the same learning rate and the same number of epochs.  Record the loss before each update, in both implementations.
Plot the two loss curves on one figure, and plot their difference.  Report the largest difference between the final parameters of the two networks, and the test accuracy of each.  They should agree to rounding error.
If they do not, check each of these in turn: the initial weights, the learning rate, whether the loss is a mean or a sum over samples, and at which point in the loop the loss is recorded. (5 points)

**(d) Compare the speed.**  Time one epoch (forward pass, backward pass, and update on the whole training set) of your numpy network and of the PyTorch network, for $h = 4, 64,$ and $512$ hidden units, and for training sets of $n = 450$ samples and $n = 20000$ samples (repeat the rows of the data to get 20000).
For each timing, run one epoch first as a warm-up, then average over at least 20 epochs (use `time.perf_counter()`).  Report a table of milliseconds per epoch for both implementations and their ratio.  Which is faster in each of the six cases?  Explain the pattern.
*(Optional, if you have a GPU:)* repeat the timings with the model and data on the GPU, calling `torch.cuda.synchronize()` before reading the timer.  What does the GPU do for the smallest case?  For the largest? (7 points; the GPU repeat is optional and earns no extra credit)

**(e) Discussion.**  In a few sentences each: (1) What did PyTorch do for you that your numpy code did not?  (2) What did writing the backpropagation yourself give you?  (3) Your XOR unit test in Problem 2(a) passed.  Why was that not enough to show that your gradients were right? (4 points)